# Prepare ML Training Data

**Project**: FLAPS  
**Purpose**: Combine cleaned flight data with weather features from Sydney and Melbourne for ML training.

---

## Table of Contents

1. [Setup](#1-setup)
2. [Load Data](#2-load-data)
3. [Data Exploration & Time Period Analysis](#3-data-exploration--time-period-analysis)
4. [Combine Data](#4-combine-data)
5. [Final Dataset Preparation](#5-final-dataset-preparation)
6. [Export](#6-export)
7. [Summary](#7-summary)

## 1. Setup

In [1]:
import pandas as pd

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [2]:
# File paths
FLIGHT_DATA_PATH = "../data/processed/flight_data_cleaned_syd_mel.csv"
WEATHER_SYD_PATH = "../data/processed/features_syd.csv"
WEATHER_MEL_PATH = "../data/processed/features_mel.csv"
OUTPUT_PATH = "../data/processed/ml_training_data_syd_mel.csv"

## 2. Load Data

In [4]:
# Load flight data
df_flights = pd.read_csv(FLIGHT_DATA_PATH)
print(f"Flight data loaded: {df_flights.shape}")
print(f"Columns: {df_flights.columns.tolist()}")
print(f"\n{'='*80}")
df_flights.head()

KeyboardInterrupt: 

In [ ]:
# Load Sydney weather features
df_weather_syd = pd.read_csv(WEATHER_SYD_PATH)
print(f"Sydney weather features loaded: {df_weather_syd.shape}")
print(f"Columns: {df_weather_syd.columns.tolist()}")
print(f"\n{'='*80}")
df_weather_syd.head()

In [ ]:
# Load Melbourne weather features
df_weather_mel = pd.read_csv(WEATHER_MEL_PATH)
print(f"Melbourne weather features loaded: {df_weather_mel.shape}")
print(f"Columns: {df_weather_mel.columns.tolist()}")
print(f"\n{'='*80}")
df_weather_mel.head()

## 3. Data Exploration & Time Period Analysis

The flight and weather data have differnt date ranges. Check date ranges for each dataset and identify the overlapping time period to maximize available training data.

In [ ]:
# Check date ranges for each dataset
print("Date ranges:")
print(f"  Flight data:      {df_flights['year_month'].min()} to {df_flights['year_month'].max()}")
print(f"  Sydney weather:   {df_weather_syd['year_month'].min()} to {df_weather_syd['year_month'].max()}")
print(f"  Melbourne weather: {df_weather_mel['year_month'].min()} to {df_weather_mel['year_month'].max()}")
print(f"\n{'='*80}")

# Find overlapping period by finding common months across all three datasets
flight_months = set(df_flights['year_month'].unique())
syd_months = set(df_weather_syd['year_month'].unique())
mel_months = set(df_weather_mel['year_month'].unique())

common_months = flight_months & syd_months & mel_months
common_months_sorted = sorted(common_months)

print(f"\nOverlapping period:")
print(f"  Flight data unique months: {len(flight_months)}")
print(f"  Sydney weather unique months: {len(syd_months)}")
print(f"  Melbourne weather unique months: {len(mel_months)}")
print(f"  Common months (all datasets): {len(common_months)}")
print(f"\n{'='*80}")

if common_months_sorted:
    print(f"\nOverlapping period: {common_months_sorted[0]} to {common_months_sorted[-1]}")
else:
    print("WARNING: No overlapping months found!")

In [ ]:
# Check for any flight months missing weather data
missing_syd = flight_months - syd_months
missing_mel = flight_months - mel_months

if missing_syd:
    print(f"WARNING: Flight months missing Sydney weather data: {sorted(missing_syd)}")
else:
    print("All flight months have Sydney weather data")

if missing_mel:
    print(f"WARNING: Flight months missing Melbourne weather data: {sorted(missing_mel)}")
else:
    print("All flight months have Melbourne weather data")

print(f"\n{'='*80}")

In [ ]:
# View route combinations in flight data
print("Route combinations in flight data:")
print(df_flights.groupby(['departing_port', 'arriving_port']).size().reset_index(name='count'))
print(f"\n{'='*80}")

## 4. Combine Data

Merge flight data with weather features from both departure and arrival airports.

### Merging Approach

For each flight record, we merge weather data from **both** airports:

1. **Departure airport weather** (suffix: `_dep`): Weather conditions at the departure airport may affect departure delays and overall flight performance.

2. **Arrival airport weather** (suffix: `_arr`): Weather conditions at the arrival airport may affect arrival delays, diversions, or holding patterns.

The Sydney-Melbourne route goes both ways (bidirectional):
- **SYD -> MEL**: Departure weather = Sydney, Arrival weather = Melbourne
- **MEL -> SYD**: Departure weather = Melbourne, Arrival weather = Sydney

In [ ]:
# Get weather feature columns (exclude year_month which is the join key)
weather_feature_cols = [col for col in df_weather_syd.columns if col != 'year_month']
print(f"Weather feature columns ({len(weather_feature_cols)}):")
print(weather_feature_cols)
print(f"\n{'='*80}")

# Prepare weather dataframes, adding suffixes for departure
df_weather_syd_dep = df_weather_syd.copy()
df_weather_syd_dep.columns = ['year_month'] + [f"{col}_dep" for col in weather_feature_cols]

df_weather_mel_dep = df_weather_mel.copy()
df_weather_mel_dep.columns = ['year_month'] + [f"{col}_dep" for col in weather_feature_cols]

# Repeat for arrival
df_weather_syd_arr = df_weather_syd.copy()
df_weather_syd_arr.columns = ['year_month'] + [f"{col}_arr" for col in weather_feature_cols]

df_weather_mel_arr = df_weather_mel.copy()
df_weather_mel_arr.columns = ['year_month'] + [f"{col}_arr" for col in weather_feature_cols]

print("Weather dataframes prepared with _dep and _arr suffixes")

In [ ]:
# Extract flights by departing port
df_from_syd = df_flights[df_flights['departing_port'] == 'Sydney'].copy()
df_from_mel = df_flights[df_flights['departing_port'] == 'Melbourne'].copy()

print(f"Flights from Sydney: {len(df_from_syd)}")
print(f"Flights from Melbourne: {len(df_from_mel)}")
print(f"Total: {len(df_from_syd) + len(df_from_mel)}")
print(f"\n{'='*80}")

In [ ]:
# Merge SYD to MEL flights with Sydney departure weather and Melbourne arrival weather
df_from_syd = df_from_syd.merge(df_weather_syd_dep, on='year_month', how='left')
df_from_syd = df_from_syd.merge(df_weather_mel_arr, on='year_month', how='left')

print(f"Sydney departures after merge: {df_from_syd.shape}")
print(f"Null values in departure weather: {df_from_syd[[c for c in df_from_syd.columns if c.endswith('_dep')]].isnull().sum().sum()}")
print(f"Null values in arrival weather: {df_from_syd[[c for c in df_from_syd.columns if c.endswith('_arr')]].isnull().sum().sum()}")

In [ ]:
# Merge MEL to SYD flights with Melbourne departure weather and Sydney arrival weather
df_from_mel = df_from_mel.merge(df_weather_mel_dep, on='year_month', how='left')
df_from_mel = df_from_mel.merge(df_weather_syd_arr, on='year_month', how='left')

print(f"Melbourne departures after merge: {df_from_mel.shape}")
print(f"Null values in departure weather: {df_from_mel[[c for c in df_from_mel.columns if c.endswith('_dep')]].isnull().sum().sum()}")
print(f"Null values in arrival weather: {df_from_mel[[c for c in df_from_mel.columns if c.endswith('_arr')]].isnull().sum().sum()}")

In [ ]:
# Combine both directions
df_combined = pd.concat([df_from_syd, df_from_mel], ignore_index=True)

print(f"Combined dataset shape: {df_combined.shape}")
print(f"Original flight data shape: {df_flights.shape}")
print(f"\n{'='*80}")

# Verify row count matches
if len(df_combined) == len(df_flights):
    print("Row count verification: PASSED")
else:
    print(f"WARNING: Row count mismatch! Expected {len(df_flights)}, got {len(df_combined)}")

In [ ]:
# Check for any null values after merge
null_counts = df_combined.isnull().sum()
cols_with_nulls = null_counts[null_counts > 0]

if len(cols_with_nulls) > 0:
    print("Columns with null values after merge:")
    print(cols_with_nulls)
    print(f"\n{'='*80}")
    
    # Show which year_months have missing data
    null_rows = df_combined[df_combined.isnull().any(axis=1)]
    print(f"Rows with null values: {len(null_rows)}")
    print(f"Year months with missing data: {null_rows['year_month'].unique()}")
else:
    print("No null values found after merge")

## 5. Final Dataset Preparation

Organize columns and prepare the final dataset for ML training.

In [ ]:
# Define column categories (from inspecting flight data frames)
flight_cols = ['departing_port', 'arriving_port', 'airline', 'month', 'year_month', 'year',
               'sectors_scheduled', 'sectors_flown', 'arrivals_on_time', 'arrivals_delayed',
               'cancellations', 'cancellations_pct']

target_cols = ['delay_rate', 'is_high_delay']

weather_dep_cols = [c for c in df_combined.columns if c.endswith('_dep')]
weather_arr_cols = [c for c in df_combined.columns if c.endswith('_arr')]

print(f"Flight metadata columns: {len(flight_cols)}")
print(f"Target columns: {len(target_cols)}")
print(f"Departure weather features: {len(weather_dep_cols)}")
print(f"Arrival weather features: {len(weather_arr_cols)}")
print(f"Total columns: {len(flight_cols) + len(target_cols) + len(weather_dep_cols) + len(weather_arr_cols)}")
print(f"\n{'='*80}")

# Reorder columns: flight info -> targets -> departure weather -> arrival weather
column_order = flight_cols + target_cols + sorted(weather_dep_cols) + sorted(weather_arr_cols)
df_final = df_combined[column_order].copy()

# Sort by date, airline, and route for consistency
df_final = df_final.sort_values(['year_month', 'airline', 'departing_port']).reset_index(drop=True)

print(f"Final dataset shape: {df_final.shape}")
print(f"\n{'='*80}")
df_final.head()

In [ ]:
# Summary statistics for target variables
print("Target variable summary:")
print(f"\ndelay_rate:")
print(df_final['delay_rate'].describe())
print(f"\nis_high_delay distribution:")
print(df_final['is_high_delay'].value_counts())
print(f"\n{'='*80}")

In [ ]:
# Display info about ML-ready features
print("\nFeature columns available for ML:")
print("\nDeparture airport weather features (_dep):")
for col in sorted(weather_dep_cols):
    print(f"  - {col}")

print("\nArrival airport weather features (_arr):")
for col in sorted(weather_arr_cols):
    print(f"  - {col}")

## 6. Export

Save the combined dataset to the processed data folder.

In [ ]:
# Save to CSV
df_final.to_csv(OUTPUT_PATH, index=False)
print(f"ML training data saved to: {OUTPUT_PATH}")
print(f"\n{'='*80}")


## 7. Summary

**Data Combination Steps Completed:**

1. Loaded cleaned flight data (Sydney-Melbourne route, both ways)
2. Loaded weather features for Sydney and Melbourne airports
3. Verified date range overlap across all datasets
4. Merged weather data based on flight direction:
   - SYD -> MEL: Sydney weather as departure, Melbourne weather as arrival
   - MEL -> SYD: Melbourne weather as departure, Sydney weather as arrival
5. Added `_dep` and `_arr` suffixes to distinguish weather features by airport role
6. Organized columns and saved final dataset

**Output:**
- `data/processed/ml_training_data_syd_mel.csv`

**Next Step**: Exploratory Data Analysis (EDA) on combined dataset